In [6]:
%load_ext autoreload
%autoreload 2

import xarray as xr
import torch
import yaml
import sys
from pathlib import Path
root_dir = Path.cwd().parent   
sys.path.append(str(root_dir))

from data.dataset import ERA5Dataset
from data.dataloader import *

print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA version (runtime):", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")

from huggingface_hub import snapshot_download

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Torch version: 2.3.1
CUDA available: True
CUDA version (runtime): 12.1
GPU: Quadro T1000 with Max-Q Design


Load the configurations and the dataset.  

In [2]:
# Load config
config_path = Path.cwd().parent / "utils" / "default_config.yaml"
with open(config_path, "r") as f:
    config = yaml.safe_load(f)
print("Config loaded!")

# snapshot_download(repo_id=config["data"]["repo_id"],
#                   repo_type='dataset',
#                   local_dir=config["data"]["local_dir"], 
#                   allow_patterns="*")

Config loaded!


Get the dataloaders for each split, ready to plug into the ML pipeline.

In [7]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

train_loader, val_loader, test_loader = get_dataloaders(
    config=config["data"],
    batch_size=config["training"]["batch_size"], 
    num_workers=config["training"]["num_workers"],
    # device=device
)

Using device: cuda
Creating datasets...
Dataset sizes -> Train: 87400, Val: 17488, Test: 17488
Creating dataloaders...
Dataloaders ready!


Test out the first iteration.

In [ ]:
# X, y = next(iter(train_loader)) 
# print("Device:", X.device, y.device)

# Fetch first batch
batch = next(iter(train_loader))  # PyG Batch
batch = batch.to(device)          # move entire graph batch to GPU

print(batch)
print("x:", batch.x.shape, batch.x.device)
print("edge_index:", batch.edge_index.shape, batch.edge_index.device)
print("y:", batch.y.shape, batch.y.device)


RuntimeError: Caught RuntimeError in DataLoader worker process 0.
Original Traceback (most recent call last):
  File "c:\Users\yaren\miniconda3\envs\weather-cast\lib\site-packages\torch\utils\data\_utils\worker.py", line 308, in _worker_loop
    data = fetcher.fetch(index)  # type: ignore[possibly-undefined]
  File "c:\Users\yaren\miniconda3\envs\weather-cast\lib\site-packages\torch\utils\data\_utils\fetch.py", line 51, in fetch
    data = [self.dataset[idx] for idx in possibly_batched_index]
  File "c:\Users\yaren\miniconda3\envs\weather-cast\lib\site-packages\torch\utils\data\_utils\fetch.py", line 51, in <listcomp>
    data = [self.dataset[idx] for idx in possibly_batched_index]
  File "c:\Users\yaren\OneDrive\Desktop\Yaren's\TU Delft\Masters\Q5\Graph Machine Learning\graph-ml-g12\data\dataset.py", line 89, in __getitem__
    graph = to_spatio_temporal_graph(X, y, self.H, self.W)
  File "c:\Users\yaren\OneDrive\Desktop\Yaren's\TU Delft\Masters\Q5\Graph Machine Learning\graph-ml-g12\